# 107 — 5-Tier Delta-ML

**Motivation:** nb104 uses 3 similarity tiers [HIGH/MED/LOW]. Adding two more tiers — VERY_HIGH at the top (densest SAR signal) and VERY_LOW at the bottom (widest fallback) — may capture additional precision at the extremes.

**Strategy:**
1. 5 tiers: VERY_HIGH [0.75,0.90], HIGH [0.60,0.75), MED [0.45,0.60), LOW [0.35,0.45), VERY_LOW [0.25,0.35)
2. Tune each tier's LGBM hyperparams separately based on pair count (fewer pairs → simpler model)
3. Tiered inference: use highest available tier; VERY_LOW only as fallback when no better tier exists
4. Compare 5-tier vs 3-tier (nb104) OOF RAE

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
# 5 tiers: (name, lo, hi)
TIERS = [
    ("VERY_HIGH", 0.75, 0.90),
    ("HIGH",      0.60, 0.75),
    ("MED",       0.45, 0.60),
    ("LOW",       0.35, 0.45),
    ("VERY_LOW",  0.25, 0.35),
]

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
from pxr.chem import compute_physchem
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Computing physchem...


Train 4,139  Test 513  Cliffs 0


In [4]:
# --- Compute pairwise Tanimoto ---
SIM_GLOBAL_LO = TIERS[-1][1]  # lowest tier lower bound (VERY_LOW lo = 0.25)
SIM_GLOBAL_HI = TIERS[0][2]   # highest tier upper bound (VERY_HIGH hi = 0.90)
MAX_PAIRS_PER_TIER = 200_000
print("Computing pairwise Tanimoto (train x train)...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)
print(f"Tanimoto matrix shape: {tanimoto_tr.shape}", flush=True)

Computing pairwise Tanimoto (train x train)...


Tanimoto matrix shape: (4139, 4139)


In [5]:
# --- Feature helpers ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_delta_feats(fp_anchor, fp_query, sim_col, anchor_pec50, phys_diff):
    fp_common = np.minimum(fp_anchor, fp_query).astype(np.float32)
    fp_diff   = np.abs(fp_anchor - fp_query).astype(np.float32)
    c64 = compress_fp(fp_common)
    d64 = compress_fp(fp_diff)
    return np.hstack([c64, d64, sim_col, anchor_pec50[:,None], phys_diff])

# Per-tier LGBM hyperparams: tune complexity based on pair count
# VERY_HIGH: fewest pairs -> more trees, simpler leaves
# VERY_LOW: many pairs -> fewer trees, more leaves
TIER_LGBM = {
    "VERY_HIGH": dict(n_estimators=800, num_leaves=31, learning_rate=0.04,
                      min_child_samples=5,  subsample=0.9, colsample_bytree=0.8,
                      reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4),
    "HIGH":      dict(n_estimators=700, num_leaves=47, learning_rate=0.05,
                      min_child_samples=8,  subsample=0.85, colsample_bytree=0.75,
                      reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4),
    "MED":       dict(n_estimators=600, num_leaves=63, learning_rate=0.05,
                      min_child_samples=10, subsample=0.8, colsample_bytree=0.7,
                      reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4),
    "LOW":       dict(n_estimators=500, num_leaves=63, learning_rate=0.05,
                      min_child_samples=15, subsample=0.8, colsample_bytree=0.7,
                      reg_alpha=0.1,  reg_lambda=0.15, random_state=SEED, verbose=-1, n_jobs=4),
    "VERY_LOW":  dict(n_estimators=400, num_leaves=47, learning_rate=0.05,
                      min_child_samples=20, subsample=0.75, colsample_bytree=0.65,
                      reg_alpha=0.15, reg_lambda=0.2, random_state=SEED, verbose=-1, n_jobs=4),
}

rng = np.random.default_rng(SEED)
tier_models = {}  # tier_name -> fitted LGBMRegressor
tier_stats  = {}

for tier_name, t_lo, t_hi in TIERS:
    if tier_name == "VERY_HIGH":
        # Top tier: include upper bound
        i_idx, j_idx = np.where((tanimoto_tr >= t_lo) & (tanimoto_tr <= t_hi))
    else:
        i_idx, j_idx = np.where((tanimoto_tr >= t_lo) & (tanimoto_tr < t_hi))
    mask_upper = i_idx < j_idx
    i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
    print(f"Tier {tier_name} [{t_lo:.2f},{t_hi:.2f}): {len(i_idx):,} pairs", flush=True)
    if len(i_idx) == 0:
        print(f"  WARNING: no pairs for tier {tier_name}, skipping", flush=True)
        continue
    if len(i_idx) > MAX_PAIRS_PER_TIER:
        sel = rng.choice(len(i_idx), MAX_PAIRS_PER_TIER, replace=False)
        i_idx, j_idx = i_idx[sel], j_idx[sel]
        print(f"  Downsampled to {MAX_PAIRS_PER_TIER:,}", flush=True)
    sims_ij = tanimoto_tr[i_idx, j_idx][:,None]
    phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
    F_ij = make_delta_feats(fps_tr[i_idx], fps_tr[j_idx], sims_ij, y_tr[i_idx], phys_diff_ij)
    F_ji = make_delta_feats(fps_tr[j_idx], fps_tr[i_idx], sims_ij, y_tr[j_idx], -phys_diff_ij)
    F_all = np.vstack([F_ij, F_ji])
    y_all = np.concatenate([y_tr[j_idx]-y_tr[i_idx], y_tr[i_idx]-y_tr[j_idx]])
    delta_mag = float(np.abs(y_all).mean())
    print(f"  avg |delta| = {delta_mag:.3f}", flush=True)
    tier_stats[tier_name] = {"n_pairs": len(i_idx), "avg_delta_mag": delta_mag}
    m = lgb.LGBMRegressor(**TIER_LGBM[tier_name])
    m.fit(F_all, y_all, callbacks=[lgb.log_evaluation(-1)])
    tier_models[tier_name] = m
    print(f"  Tier {tier_name} model trained.", flush=True)

print(f"\nTrained {len(tier_models)} tier models: {list(tier_models.keys())}")
print(pd.DataFrame(tier_stats).T.round(3))

Tier VERY_HIGH [0.75,0.90): 8 pairs


  avg |delta| = 0.736


  Tier VERY_HIGH model trained.


Tier HIGH [0.60,0.75): 109 pairs


  avg |delta| = 0.730


...

  Tier LOW model trained.


Tier VERY_LOW [0.25,0.35): 69,119 pairs


  avg |delta| = 0.971


  Tier VERY_LOW model trained.



Trained 5 tier models: ['VERY_HIGH', 'HIGH', 'MED', 'LOW', 'VERY_LOW']
           n_pairs  avg_delta_mag
VERY_HIGH      8.0          0.736
HIGH         109.0          0.730
MED          910.0          0.756
LOW         4150.0          0.870
VERY_LOW   69119.0          0.971


In [6]:
# --- 5-tier multi-template prediction ---
K_NEIGHBORS = 8

def tiered5_delta_predict(fps_query, fps_ref, y_ref, phys_query, phys_ref,
                           sim_matrix, tier_models, tiers, direct_preds, k=K_NEIGHBORS):
    """
    For each query:
    1. Try VERY_HIGH tier first; if >= 1 template found, use that tier
    2. Fall through tiers: HIGH -> MED -> LOW -> VERY_LOW
    3. VERY_LOW used only as last-resort fallback before direct prediction
    4. Within a tier, weight by sim^2
    Returns: (preds, tier_used, n_templates)
    """
    N = len(fps_query)
    preds = np.full(N, np.nan)
    tier_used = np.full(N, -1, dtype=int)
    n_templates = np.zeros(N, dtype=int)

    for qi in range(N):
        sim_row = sim_matrix[qi]
        found = False
        for tier_i, (tier_name, t_lo, t_hi) in enumerate(tiers):
            if tier_name not in tier_models:
                continue
            if tier_i == 0:  # VERY_HIGH: include upper bound
                cand_mask = (sim_row >= t_lo) & (sim_row <= t_hi)
            else:
                cand_mask = (sim_row >= t_lo) & (sim_row < t_hi)
            cand_idx = np.where(cand_mask)[0]
            if len(cand_idx) == 0:
                continue
            top_k = np.argsort(-sim_row[cand_idx])[:k]
            cand_idx = cand_idx[top_k]
            cand_sims = sim_row[cand_idx]
            n_templates[qi] = len(cand_idx)
            tier_used[qi] = tier_i
            fp_q_rep = np.tile(fps_query[qi:qi+1], (len(cand_idx), 1))
            fp_refs = fps_ref[cand_idx]
            sims_col = cand_sims[:,None]
            anc_pec50 = y_ref[cand_idx]
            phys_d = phys_query[qi:qi+1] - phys_ref[cand_idx]
            F_k = make_delta_feats(fp_refs, fp_q_rep, sims_col, anc_pec50, phys_d)
            delta_k = tier_models[tier_name].predict(F_k)
            template_preds = y_ref[cand_idx] + delta_k
            weights = cand_sims ** 2
            preds[qi] = np.average(template_preds, weights=weights)
            found = True
            break
        if not found:
            preds[qi] = direct_preds[qi]

    return preds, tier_used, n_templates

print(f"5-tier predict function ready (tiers={[t[0] for t in TIERS]}, K={K_NEIGHBORS})")

5-tier predict function ready (tiers=['VERY_HIGH', 'HIGH', 'MED', 'LOW', 'VERY_LOW'], K=8)


In [7]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_tiered5 = np.full(len(y_tr), np.nan)
oof_direct  = np.full(len(y_tr), np.nan)
tier_counts = {t[0]: 0 for t in TIERS}
tier_counts["fallback"] = 0

for fold, (tr_idx, va_idx) in enumerate(splits):
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    preds_t, tier_u, n_tmpl = tiered5_delta_predict(
        fps_va, fps_ft, y_tr[tr_idx], phys_arr[va_idx], phys_arr[tr_idx],
        sim_vf, tier_models, TIERS, oof_direct[va_idx]
    )
    oof_tiered5[va_idx] = preds_t

    for ti, (tname, _, _) in enumerate(TIERS):
        tier_counts[tname] += int((tier_u == ti).sum())
    tier_counts["fallback"] += int((tier_u == -1).sum())

    tier_str = "/".join(f"{t[0]}:{(tier_u==ti).sum()}" for ti,t in enumerate(TIERS))
    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_t   = rae(y_tr[va_idx], oof_tiered5[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  tiered5={r_t:.4f}  "
          f"avg_tmpl={float(n_tmpl.mean()):.1f}  FB:{(tier_u==-1).sum()}",
          flush=True)

print(f"\nTotal tier usage across all folds: {tier_counts}")
m_dir     = full_metrics(y_tr, oof_direct,   cliff_pairs, "direct_lgbm")
m_tiered5 = full_metrics(y_tr, oof_tiered5,  cliff_pairs, "tiered5_delta")


=== Scaffold 5-fold CV ===


  fold 1  direct=0.4982  tiered5=0.2632  avg_tmpl=3.4  FB:16


  fold 2  direct=0.5759  tiered5=0.2780  avg_tmpl=3.5  FB:17


  fold 3  direct=0.6021  tiered5=0.3052  avg_tmpl=3.8  FB:13


  fold 4  direct=0.5665  tiered5=0.3050  avg_tmpl=3.5  FB:13


  fold 5  direct=0.6033  tiered5=0.3044  avg_tmpl=3.6  FB:24



Total tier usage across all folds: {'VERY_HIGH': 2, 'HIGH': 74, 'MED': 823, 'LOW': 1656, 'VERY_LOW': 1501, 'fallback': 83}
  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [tiered5_delta] RAE=0.2888 MAE=0.2628 R2=0.8385 r=0.9195 rho=0.8916 tau=0.7435


In [8]:
# --- Compare vs nb104 (3-tier) ---
nb104_oof_path = DATA_PROCESSED / "oof_delta_similarity_tiers.npy"
if nb104_oof_path.exists():
    oof_3tier = np.load(nb104_oof_path)
    m_3tier = full_metrics(y_tr, oof_3tier, cliff_pairs, "nb104_3tier")
    print(f"\n3-tier (nb104) OOF RAE: {m_3tier['RAE']:.4f}")
    print(f"5-tier (nb107) OOF RAE: {m_tiered5['RAE']:.4f}")
    print(f"Delta: {m_tiered5['RAE'] - m_3tier['RAE']:+.4f}")

# --- Blend sweep ---
best_alpha, best_rae_v = 0.0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_tiered5 + (1-alpha)*oof_direct
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_alpha = r, alpha

oof = best_alpha*oof_tiered5 + (1-best_alpha)*oof_direct
m_blend = full_metrics(y_tr, oof, cliff_pairs, f"blend(a={best_alpha:.1f})")
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae_v:.4f}")

  [nb104_3tier] RAE=0.2772 MAE=0.2522 R2=0.8300 r=0.9115 rho=0.8831 tau=0.7437

3-tier (nb104) OOF RAE: 0.2772
5-tier (nb107) OOF RAE: 0.2888
Delta: +0.0116
  alpha=0.0  RAE=0.5643
  alpha=0.1  RAE=0.5318
  alpha=0.2  RAE=0.5000
  alpha=0.3  RAE=0.4691
  alpha=0.4  RAE=0.4391
  alpha=0.5  RAE=0.4100
  alpha=0.6  RAE=0.3824
  alpha=0.7  RAE=0.3563
  alpha=0.8  RAE=0.3316
  alpha=0.9  RAE=0.3086
  alpha=1.0  RAE=0.2888
  [blend(a=1.0)] RAE=0.2888 MAE=0.2628 R2=0.8385 r=0.9195 rho=0.8916 tau=0.7435

Best blend alpha=1.0  OOF RAE=0.2888


In [9]:
# --- Final test predictions ---
print("\nFitting final direct LGBM on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)
phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)

print("Running 5-tier delta on test...", flush=True)
te_tiered5, te_tier_u, n_tmpl_te = tiered5_delta_predict(
    fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, tier_models, TIERS, te_direct
)
tier_str = "/".join(f"{t[0]}:{(te_tier_u==ti).sum()}" for ti,t in enumerate(TIERS))
print(f"Test tier usage: {tier_str}/FB:{(te_tier_u==-1).sum()}")

te_preds = best_alpha*te_tiered5 + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_delta_5tiers.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_5tiers.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"107_delta_5tiers.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb107 OOF RAE = {m_blend['RAE']:.4f} ***")


Fitting final direct LGBM on all train...


Running 5-tier delta on test...


Test tier usage: VERY_HIGH:4/HIGH:87/MED:369/LOW:50/VERY_LOW:3/FB:0
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\107_delta_5tiers.csv
Test: min=3.03 med=4.95 max=6.69

*** nb107 OOF RAE = 0.2888 ***
